# Phase 3 (first pass) — SMI Calculation + Reliability (DREADD saline/DCZ cohort)

Scope for this pass: whole-session SMI computation for Track A (tracked
cells, from Phase 2's TRACKING_GROUPS) and Track B (all cells passing QC,
per session, independent), plus a reliability diagnostic. **DCZ
pharmacokinetic time-binning (originally 3b) is deliberately deferred** —
this pass uses full-session SMI only.

Reuses the existing SMI machinery via import rather than reimplementing it —
`helper/SMI_Calculation.py`'s `analyze_spatial_modulation_improved` (the
actual SMI math) and `helper/SpatialModulationIndexLayerSpecific.py`'s
layer-bucketing code both just take a `layer_cells` dict as input; neither
cares how that dict was produced. So the only genuinely new code needed is:
(a) loading Phase 1's curve-based layer assignment in place of the old
`identify_layers()` call, and (b) the Track A/B population logic.

**Decisions locked in:**
- Reliability mask for SMI: `combined_reliable` (from `preproc.h5`) — same
  convention `Run_SMI_Layer_Analysis` already uses for every other cohort,
  and its extra peak-location-stability check is directly relevant to SMI's
  validity.
- Split-half reliability needs no new computation — `preproc.h5`'s existing
  `avg_cc` field already *is* the real odd/even correlation (traced through
  `test_cell_reliability_improved`'s shuffle loop to confirm this).
- New SMI output uses a distinct filename (`*_smi_results_dreadd.h5`) so it
  never collides with any pre-existing `smi_results.h5` for these sessions.
- Known caveat, not fixed in this pass: whole-session reliability comparisons
  between saline (short) and DCZ (long) recordings will have a trial-count
  confound baked in, since `combined_reliable`'s thresholds aren't
  duration-adjusted. Interpret saline-vs-DCZ reliability-rate differences
  cautiously until the deferred time-binning work addresses this.

Built incrementally, one function at a time. Consolidated into
`3.SMICalculation.py` only once everything here works end-to-end on real data.

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation")

import os
import re
import glob
import numpy as np
import h5py
import pandas as pd
import matplotlib
matplotlib.use('Qt5Agg')  # interactive popups (checkbox pickers etc.) need a real GUI backend, not the notebook's inline default
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.widgets import CheckButtons, Button

rcParams['legend.fontsize'] = 20
rcParams['axes.labelsize'] = 20
rcParams['axes.titlesize'] = 25
rcParams['xtick.labelsize'] = 20
rcParams['ytick.labelsize'] = 20

# Existing pipeline code, reused via import -- not modified.
from helper import files
from helper import SMI_Calculation as SMI
from helper.SpatialModulationIndexLayerSpecific import SpatialModulationIndexLayerSpecific as SMI_Layer
from helper.SMICalculation_LayerSpecific_SingleRecording import filter_onset_response_cells
from helper.ReliabilityTesting import evaluate_pattern_similarity_improved, improved_activity_threshold_check

MICRONS_PER_PIXEL = 1.08952017715202  # V1_prism_DREADD animals (JSY090, JSY093)

# One real DREADD session to develop and sanity-check each function against.
TEST_SESSION_DIR = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\260719_JSY_JSY090_LongitudinalImaging_DREADD_Day1\TSeries-07192026-0941-001"
TEST_PLANE0 = os.path.join(TEST_SESSION_DIR, 'suite2p', 'plane0')
TEST_ANIMAL_DIR = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD"


## Function 3.1 — `load_session_layer_cells_for_smi`

Loads Phase 1's curve-based layer assignment for one session and converts
it into the `{layer_name: indices}` dict shape the existing layer-bucketing
code (`SpatialModulationIndexLayerSpecific.run_layer_SMI_analysis` /
`analyze_layer_specific_smi_from_existing_results`) expects — the same role
`identify_layers()` used to play in `Run_SMI_Layer_Analysis`, just reading a
precomputed assignment instead of computing a fresh one.

Verified alignment before building this: `layer_codes`' position *i* (from
Phase 1's `.h5`) refers to the same physical cell as position *i* in
`preproc.h5`'s arrays — both are built from `stat[iscell[:,0]==1]`
(`TwoP.find_files`), the identical boolean-mask ordering. So no index
translation is needed here, only a straight reshape from flat codes to a
dict of index arrays.

Includes its own small `find_layer_curve_path` (same logic as Phase 2's,
reimplemented here rather than imported — module filenames starting with a
digit can't be `import`ed via a normal statement).

- **Input:** `plane0_path`, `layer_names=('L2/3', 'L4', 'L5', 'L6')` (must
  match the order Phase 1 saved).
- **Output:** `layer_cells`, `{layer_name: numpy.ndarray of cell indices}`.

In [ ]:
def find_layer_curve_path(plane0_path, prefer_averaged=True):
    """
    Locate a session's Phase 1 layer-curve-results file, given its
    suite2p/plane0 path. Same logic as 2.CellTracking.py's function of the
    same name, reimplemented here since neither Phase 1 nor Phase 2's
    module filenames can be imported via a normal Python import statement.
    """
    tseries_dir = os.path.dirname(os.path.dirname(str(plane0_path)))
    averaged_matches = glob.glob(os.path.join(tseries_dir, '*_layer_curve_results_averaged.h5'))
    independent_matches = glob.glob(os.path.join(tseries_dir, '*_layer_curve_results.h5'))

    if prefer_averaged and averaged_matches:
        matches = averaged_matches
    elif independent_matches:
        matches = independent_matches
    elif averaged_matches:
        matches = averaged_matches
    else:
        raise FileNotFoundError(f"No *_layer_curve_results(_averaged).h5 found in {tseries_dir} "
                                 "-- has Phase 1 been run for this session?")

    if len(matches) > 1:
        print(f"WARNING: multiple matches in {tseries_dir}, using {matches[0]}")
    return matches[0]


def load_session_layer_cells_for_smi(plane0_path, layer_names=('L2/3', 'L4', 'L5', 'L6')):
    """
    Load Phase 1's curve-based layer assignment for one session, converted
    to {layer_name: indices} -- the shape identify_layers() used to produce.

    Parameters
    ----------
    plane0_path : str
        Path to the session's suite2p/plane0 folder.
    layer_names : tuple of str
        Expected layer name order (must match what Phase 1 saved).

    Returns
    -------
    layer_cells : dict
        {layer_name: numpy.ndarray of cell indices}, indices are positions
        in the iscell-filtered cell list -- same convention as
        preproc.h5's arrays.
    """
    layer_curve_path = find_layer_curve_path(plane0_path)

    with h5py.File(layer_curve_path, 'r') as f:
        layer_codes = f['layer_codes'][:]
        saved_layer_names = tuple(n.decode() if isinstance(n, bytes) else n
                                   for n in f['layer_names'][:])

    if saved_layer_names != layer_names:
        print(f"NOTE: saved layer_names {saved_layer_names} differ in order from "
              f"the requested {layer_names} -- using the saved order.")

    layer_cells = {
        name: np.where(layer_codes == code)[0]
        for code, name in enumerate(saved_layer_names)
    }

    print(f"Loaded layer assignment from {layer_curve_path}")
    for name, idx in layer_cells.items():
        print(f"  {name}: {len(idx)} cells")

    return layer_cells


In [ ]:
# --- Try it on the real Day1 session ---
layer_cells_test = load_session_layer_cells_for_smi(TEST_PLANE0)
print("\nTotal cells across all layers:", sum(len(v) for v in layer_cells_test.values()))


## Function 3.2 — `run_smi_analysis_session`

The per-session SMI driver -- Track B directly (all `combined_reliable`
cells, computed independently for this one session). Mirrors
`Run_SMI_Layer_Analysis`'s flow (load `preproc.h5` -> onset filtering ->
`analyze_spatial_modulation_improved` -> layer bucketing -> save), with
Step 2 (layer identification) replaced by Function 3.1's output. The bin-
center rescaling (`shifted_centers * n_bins/max(shifted_centers)`) is kept
byte-for-byte identical to the original, since `segment_distance`/
`exclude_start_cm`/`exclude_end_cm` are all calibrated in that rescaled
unit space, not raw cm.

Built as two pieces:
1. **`run_smi_analysis_session`** — the driver itself.
2. **`save_smi_results_dreadd`** — a genuinely new save function rather than
   reusing the existing `save_smi_results`: that function's layer-boundary
   section assumes each layer has a flat `(upper, lower)` tuple, which
   doesn't fit Phase 1's curve-based boundaries (a function of x, not a
   constant). Everything else about it mirrors the existing file's group/
   dataset naming for consistency, and additionally persists
   `combined_reliable`/`avg_cc`/`cohen_d` (so the reliability diagnostic,
   Function 3.3, doesn't need to reload `preproc.h5` separately) plus which
   Phase 1 layer-curve file was used, for provenance.

- **Input:** `session_dir`, `session_label=None`, onset-filter/SMI
  parameters (same defaults as `Run_SMI_Layer_Analysis`), `save_output=True`.
- **Output:** dict with `session_label`, `save_path`, `results_full`,
  `layer_results`, `layer_cells`, `analysis_reliable_cells`,
  `combined_reliable`, `avg_cc`, `cohen_d`, `med_coords`, `bin_centers`.

In [ ]:
def save_smi_results_dreadd(session_dir, session_label, result):
    """
    Save one session's DREADD SMI result to
    '{TSeries folder name}_smi_results_dreadd.h5' -- a distinct filename so
    it never collides with any pre-existing smi_results.h5 for this
    session. See markdown above for why this isn't just a call to the
    existing save_smi_results.

    The filename is built from the TSeries folder's own name, not
    session_label -- session_label can be a long, fully-disambiguated
    catalog key (see discover_animal_sessions, whose collision handling
    appends the entire raw TSeries folder name onto an already-long
    parent-folder-derived base label). Repeating that inside the filename
    on top of an already-long parent path risks exceeding Windows'
    260-character MAX_PATH limit -- hit in practice for one of the
    StationaryOpenLoop sessions. session_dir already scopes the file to
    this exact session, so the TSeries folder's own (much shorter) name is
    just as unique here. session_label is still stored as an attribute
    below, for provenance/display.

    Parameters
    ----------
    session_dir : str
    session_label : str
    result : dict
        The dict run_smi_analysis_session builds internally (before this
        save step is called).

    Returns
    -------
    save_path : str
    """
    smi_results = result['results_full']['smi_results']
    layer_results = result['layer_results']
    layer_cells = result['layer_cells']

    tseries_name = os.path.basename(session_dir)
    save_path = os.path.join(session_dir, f'{tseries_name}_smi_results_dreadd.h5')

    with h5py.File(save_path, 'w') as f:
        f.attrs['session_label'] = session_label
        f.attrs['layer_curve_source'] = result.get('layer_curve_path', '')

        global_grp = f.create_group('global_smi')
        global_grp.create_dataset('SMI_all_cells', data=smi_results['SMI'])
        global_grp.create_dataset('valid_cells_mask', data=smi_results['reliable_valid_cells'])
        global_grp.create_dataset('analysis_reliable_cells', data=result['analysis_reliable_cells'])
        global_grp.create_dataset('preferred_positions', data=smi_results['preferred_positions'])
        global_grp.create_dataset('non_preferred_positions', data=smi_results['non_preferred_positions'])
        global_grp.create_dataset('Rp', data=smi_results['Rp'])
        global_grp.create_dataset('Rn', data=smi_results['Rn'])

        # Reliability info, persisted here so Function 3.3 doesn't need to
        # reload preproc.h5 separately.
        rel_grp = f.create_group('reliability')
        rel_grp.create_dataset('combined_reliable', data=result['combined_reliable'])
        rel_grp.create_dataset('avg_cc', data=result['avg_cc'])
        rel_grp.create_dataset('cohen_d', data=result['cohen_d'])

        coords_grp = f.create_group('cell_info')
        coords_grp.create_dataset('med_coords', data=result['med_coords'])
        coords_grp.create_dataset('bin_centers', data=result['bin_centers'])

        layers_grp = f.create_group('layer_smi')
        for layer_name, cell_indices in layer_cells.items():
            safe_name = layer_name.replace('/', '_')
            layer_grp = layers_grp.create_group(safe_name)
            layer_grp.attrs['original_name'] = layer_name
            layer_grp.create_dataset('cell_indices', data=cell_indices)

            lr = layer_results.get(layer_name)
            if lr is not None:
                layer_grp.create_dataset('reliable_valid_cells', data=lr['reliable_valid_cells'])
                layer_grp.create_dataset('SMI', data=lr['SMI'])
                layer_grp.attrs['n_cells_total'] = len(cell_indices)
                layer_grp.attrs['n_cells_valid'] = len(lr['SMI'])
                layer_grp.attrs['median_smi'] = float(lr['stats']['median'])
                layer_grp.attrs['mean_smi'] = float(lr['stats']['mean'])
                layer_grp.attrs['std_smi'] = float(lr['stats']['std'])
                layer_grp.attrs['sem_smi'] = float(lr['stats']['sem'])
                if lr['preferred_positions'] is not None:
                    layer_grp.create_dataset('preferred_positions', data=lr['preferred_positions'])
                if lr['Rp'] is not None:
                    layer_grp.create_dataset('Rp', data=lr['Rp'])
                if lr['Rn'] is not None:
                    layer_grp.create_dataset('Rn', data=lr['Rn'])
            else:
                layer_grp.attrs['n_cells_total'] = len(cell_indices)
                layer_grp.attrs['n_cells_valid'] = 0

    print(f"Saved DREADD SMI results -> {save_path}")
    return save_path


def run_smi_analysis_session(session_dir, session_label=None,
                              exclude_first_bins=5, exclude_last_bins=5,
                              segment_distance=28, exclude_start_cm=15, exclude_end_cm=10,
                              smoothing_sigma=1.0, save_output=True):
    """
    Per-session SMI driver for one DREADD session. See markdown above.

    Parameters
    ----------
    session_dir : str
        Path to the TSeries folder (contains suite2p/plane0/ and *preproc*.h5).
    session_label : str, optional
    exclude_first_bins, exclude_last_bins : int
    segment_distance, exclude_start_cm, exclude_end_cm, smoothing_sigma : float
        Same defaults as Run_SMI_Layer_Analysis.
    save_output : bool

    Returns
    -------
    result : dict -- see markdown above for keys.
    """
    if session_label is None:
        session_label = os.path.basename(session_dir)

    plane0_path = os.path.join(session_dir, 'suite2p', 'plane0')

    # --- Step 1: load preproc.h5 ---
    preproc_files = glob.glob(os.path.join(session_dir, "*preproc*.h5"))
    if not preproc_files:
        raise FileNotFoundError(f"No *preproc*.h5 found in {session_dir}")
    preproc_data = files.read_h5(preproc_files[0])

    spatial_activity = preproc_data['spatial_activity']
    normalized_spatial_activity = preproc_data['norm_spatial_activity']
    bin_centers = preproc_data['bin_centers']
    combined_reliable = preproc_data['combined_reliable']
    avg_cc = preproc_data['avg_cc']
    cohen_d = preproc_data['cohen_d']
    med_coords = preproc_data['med_coords']

    n_cells, n_trials, n_bins = spatial_activity.shape
    print(f"\n{session_label}: {n_cells} cells, {n_trials} trials, {n_bins} bins")
    print(f"  combined_reliable: {np.sum(combined_reliable)} cells")

    # Same bin-center rescaling Run_SMI_Layer_Analysis uses.
    shifted_centers = bin_centers - np.min(bin_centers)
    scaled_bin_centers = shifted_centers * (np.size(bin_centers) / np.max(shifted_centers))

    # --- Step 2: layer assignment (Phase 1's curve-based output) ---
    layer_curve_path = find_layer_curve_path(plane0_path)
    layer_cells = load_session_layer_cells_for_smi(plane0_path)

    # --- Step 3: onset/reward filtering (existing, imported) ---
    non_onset_cells, rejected_info = filter_onset_response_cells(
        spatial_activity, scaled_bin_centers, combined_reliable,
        exclude_first_bins=exclude_first_bins, exclude_last_bins=exclude_last_bins
    )
    analysis_reliable_cells = combined_reliable & non_onset_cells
    print(f"  Final cells for analysis (combined_reliable & non-onset): "
          f"{np.sum(analysis_reliable_cells)}")

    # --- Step 4: SMI calculation (existing, imported, untouched) ---
    results_full = SMI.analyze_spatial_modulation_improved(
        spatial_activity, scaled_bin_centers, analysis_reliable_cells,
        avg_cc=avg_cc, cohens_d=cohen_d,
        segment_distance=segment_distance, exclude_start_cm=exclude_start_cm,
        exclude_end_cm=exclude_end_cm, smoothing_sigma=smoothing_sigma,
        data_filepath=session_dir,
    )

    # --- Step 5: layer-specific bucketing (existing, imported, untouched) ---
    # run_layer_SMI_analysis's internal plot_layer_comparison treats save_path
    # as an existing directory (os.path.join(save_path, "...png")) rather than
    # creating it -- Run_SMI_Layer_Analysis gets away with this because it
    # already created its 'SMI_Figures' dir earlier for onset-filter viz and
    # reuses that. This is a new directory name, so it must be created here.
    viz_dir = os.path.join(session_dir, 'SMI_Figures_DREADD')
    os.makedirs(viz_dir, exist_ok=True)

    valid_cells = results_full['smi_results']['reliable_valid_cells']
    layer_results, _ = SMI_Layer.run_layer_SMI_analysis(
        results_full['smi_results'], valid_cells, med_coords, layer_cells,
        normalized_spatial_activity, scaled_bin_centers,
        save_path=viz_dir
    )

    result = {
        'session_label': session_label,
        'save_path': None,
        'results_full': results_full,
        'layer_results': layer_results,
        'layer_cells': layer_cells,
        'layer_curve_path': layer_curve_path,
        'analysis_reliable_cells': analysis_reliable_cells,
        'combined_reliable': combined_reliable,
        'avg_cc': avg_cc,
        'cohen_d': cohen_d,
        'med_coords': med_coords,
        'bin_centers': scaled_bin_centers,
    }

    if save_output:
        result['save_path'] = save_smi_results_dreadd(session_dir, session_label, result)

    return result


In [ ]:
# --- Try it on the real Day1 session ---
smi_result_test = run_smi_analysis_session(TEST_SESSION_DIR)

print("\nsave_path:", smi_result_test['save_path'])
print("Layer cell counts:", {k: len(v) for k, v in smi_result_test['layer_cells'].items()})


## Function 3.3 — `build_smi_reliability_diagnostic`

Combines per-cell SMI with the already-computed `avg_cc`/`cohen_d`/
`combined_reliable` (no new computation, per the earlier simplification)
into one table, and flags two discrepancy patterns:

- **`reliable_but_low_smi`** — `combined_reliable`, with `avg_cc` in the top
  quartile of reliable cells (well above the minimum bar, i.e. a genuinely
  solid, repeatable tuning curve), but SMI below threshold anyway. Likely
  candidates: a landmark-periodicity artifact deflating `Rn` near another
  bump, or a broad/multi-peaked field that doesn't fit SMI's single-peak
  assumption well — the tuning is real, SMI's specific definition just
  doesn't capture it.
- **`high_smi_but_unreliable`** — SMI above threshold despite failing
  `combined_reliable`. An SMI value that "looks tuned" without passing the
  reliability bar — worth distrusting rather than taking at face value.

Built as two pieces: the table/flagging function, plus a scatter plot
(SMI vs. `avg_cc`, colored by `combined_reliable`, discrepancy cells
marked) as the visual complement.

- **Input:** `smi_result` (from `run_smi_analysis_session`),
  `discrepancy_smi_threshold=0.15`, `discrepancy_cc_percentile=75`.
- **Output:** `df` (one row per cell: SMI, avg_cc, cohen_d,
  combined_reliable, valid, layer, discrepancy), `summary` (counts).

In [ ]:
def build_smi_reliability_diagnostic(smi_result, discrepancy_smi_threshold=0.15,
                                      discrepancy_cc_percentile=75):
    """
    Combine per-cell SMI with existing avg_cc/cohen_d/combined_reliable into
    one diagnostic table, flagging sharp-tuning-vs-low-SMI (and the reverse)
    discrepancies. See markdown above.

    Parameters
    ----------
    smi_result : dict
        From run_smi_analysis_session.
    discrepancy_smi_threshold : float
    discrepancy_cc_percentile : float

    Returns
    -------
    df : pandas.DataFrame
    summary : dict
    """
    smi_results = smi_result['results_full']['smi_results']
    SMI_values = smi_results['SMI']
    valid_cells = smi_results['valid_cells']
    combined_reliable = smi_result['combined_reliable']
    avg_cc = smi_result['avg_cc']
    cohen_d = smi_result['cohen_d']
    layer_cells = smi_result['layer_cells']

    n_cells = len(SMI_values)
    layer_of_cell = np.full(n_cells, None, dtype=object)
    for layer_name, idx in layer_cells.items():
        layer_of_cell[idx] = layer_name

    df = pd.DataFrame({
        'cell_idx': np.arange(n_cells),
        'SMI': SMI_values,
        'avg_cc': avg_cc,
        'cohen_d': cohen_d,
        'combined_reliable': combined_reliable,
        'valid': valid_cells,
        'layer': layer_of_cell,
    })

    cc_threshold = (np.percentile(avg_cc[combined_reliable], discrepancy_cc_percentile)
                    if combined_reliable.any() else np.inf)

    reliable_low_smi = (df['combined_reliable'] & (df['avg_cc'] >= cc_threshold) &
                         (df['SMI'] < discrepancy_smi_threshold) & df['valid'])
    high_smi_unreliable = ((~df['combined_reliable']) & (df['SMI'] >= discrepancy_smi_threshold)
                            & df['valid'])

    df['discrepancy'] = None
    df.loc[reliable_low_smi, 'discrepancy'] = 'reliable_but_low_smi'
    df.loc[high_smi_unreliable, 'discrepancy'] = 'high_smi_but_unreliable'

    summary = {
        'n_cells': n_cells,
        'n_reliable_but_low_smi': int(reliable_low_smi.sum()),
        'n_high_smi_but_unreliable': int(high_smi_unreliable.sum()),
        'cc_threshold_used': float(cc_threshold),
    }

    print(f"Reliability diagnostic for {smi_result['session_label']}:")
    print(f"  reliable_but_low_smi: {summary['n_reliable_but_low_smi']} cells "
          f"(combined_reliable, avg_cc >= {cc_threshold:.3f}, SMI < {discrepancy_smi_threshold})")
    print(f"  high_smi_but_unreliable: {summary['n_high_smi_but_unreliable']} cells "
          f"(SMI >= {discrepancy_smi_threshold}, not combined_reliable)")

    return df, summary


def plot_smi_reliability_scatter(df, session_label='', save_path=None):
    """
    SMI vs. avg_cc scatter, colored by combined_reliable, discrepancy cells
    outlined -- the visual complement to build_smi_reliability_diagnostic.

    Parameters
    ----------
    df : pandas.DataFrame
        From build_smi_reliability_diagnostic.
    session_label : str
    save_path : str, optional

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    plot_df = df[df['valid']]

    fig, ax = plt.subplots(figsize=(9, 8))
    colors = np.where(plot_df['combined_reliable'], '#2ca02c', '#7f7f7f')
    ax.scatter(plot_df['avg_cc'], plot_df['SMI'], c=colors, s=25, alpha=0.6,
               label=None)

    for discrepancy_type, marker, edgecolor in [
        ('reliable_but_low_smi', 'D', 'blue'),
        ('high_smi_but_unreliable', '^', 'red'),
    ]:
        flagged = plot_df[plot_df['discrepancy'] == discrepancy_type]
        if len(flagged) > 0:
            ax.scatter(flagged['avg_cc'], flagged['SMI'], facecolors='none',
                       edgecolors=edgecolor, s=90, linewidths=1.5, marker=marker,
                       label=f'{discrepancy_type} (n={len(flagged)})')

    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('avg_cc (split-half correlation)')
    ax.set_ylabel('SMI')
    ax.set_title(f'{session_label}: SMI vs. reliability')

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ca02c', markersize=10, label='combined_reliable'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#7f7f7f', markersize=10, label='not reliable'),
    ]
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=legend_elements + handles, loc='upper left', fontsize=13)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved reliability scatter -> {save_path}")

    return fig


In [ ]:
# --- Try it on the real Day1 SMI result ---
diagnostic_df, diagnostic_summary = build_smi_reliability_diagnostic(smi_result_test)

fig = plot_smi_reliability_scatter(diagnostic_df, session_label=smi_result_test['session_label'])
plt.show()

diagnostic_df[diagnostic_df['discrepancy'].notna()]


## Function 3.3b — `diagnose_reliability_failure_reasons`

Breaks down *why* each non-`combined_reliable` cell failed. `combined_reliable
= reliable_cells & pattern_reliable & active_cells`, and `reliable_cells`
itself requires `avg_cc > min_cc_threshold` AND `avg_cc` beats its own
shuffle-null AND `cohen_d > cohen_threshold`. Of these:
- **`below_cc_floor`**, **`below_cohen_d`** — simple threshold checks on the
  already-saved `avg_cc`/`cohen_d`, no recomputation needed.
- **`inactive`**, **`pattern_unstable`** — both fully deterministic (no
  shuffling), recomputed directly from `preproc.h5`'s `spatial_activity` via
  `improved_activity_threshold_check`/`evaluate_pattern_similarity_improved`
  (imported, existing code).
- **`below_shuffle_significance_inferred`** — the one part that genuinely
  isn't recomputable without rerunning `n_shuffles=300` shuffles (per-cell
  significance vs. its own null). Inferred by elimination: a cell that
  passes both flat floors, is active, and has a stable pattern, but is
  *still* not `reliable_cells`, must have failed specifically the
  shuffle-significance check.

**Caveat:** the threshold defaults here (`min_cc_threshold=0.1`,
`cohen_threshold=0.8`, `min_pattern_corr=0.3`, `peak_distance_threshold=5`)
must match whatever `Preprocess.py` actually used for this session's
`area` — these are the `area='V1'` defaults; double-check if unsure.

- **Input:** `session_dir`, `smi_result` (from `run_smi_analysis_session`),
  the four threshold parameters, `activity_method`.
- **Output:** `df` (one row per non-reliable cell, all sub-test values +
  boolean failure flags), `summary` (counts per failure category).

In [ ]:
def diagnose_reliability_failure_reasons(session_dir, smi_result,
                                          min_cc_threshold=0.1, cohen_threshold=0.8,
                                          min_pattern_corr=0.3, peak_distance_threshold=5,
                                          activity_method='absolute_percentile'):
    """
    Break down why each non-combined_reliable cell failed. See markdown above.

    Parameters
    ----------
    session_dir : str
    smi_result : dict
        From run_smi_analysis_session.
    min_cc_threshold, cohen_threshold, min_pattern_corr, peak_distance_threshold : float
        Must match Preprocess.py's actual call for this session's area.
    activity_method : str

    Returns
    -------
    df : pandas.DataFrame
        One row per cell (all cells, flagged with combined_reliable so you
        can filter): avg_cc, cohen_d, active, pattern_reliable,
        odd_even_corr, peak_distance, and boolean failure-category columns.
    summary : dict
        Counts per failure category, among cells failing combined_reliable.
    """
    preproc_files = glob.glob(os.path.join(session_dir, "*preproc*.h5"))
    if not preproc_files:
        raise FileNotFoundError(f"No *preproc*.h5 found in {session_dir}")
    preproc_data = files.read_h5(preproc_files[0])
    spatial_activity = preproc_data['spatial_activity']

    combined_reliable = smi_result['combined_reliable']
    avg_cc = smi_result['avg_cc']
    cohen_d = smi_result['cohen_d']

    active_cells, _ = improved_activity_threshold_check(spatial_activity, method=activity_method)
    pattern_reliable, odd_even_corr, peak_distances = evaluate_pattern_similarity_improved(
        spatial_activity, min_pattern_corr, peak_distance_threshold
    )

    n_cells = len(avg_cc)
    below_cc_floor = avg_cc < min_cc_threshold
    below_cohen_d = cohen_d < cohen_threshold
    inactive = ~active_cells
    pattern_unstable = ~pattern_reliable

    # Cells failing reliable_cells despite passing both flat floors, being
    # active, and having a stable pattern -- the only remaining explanation
    # is the shuffle-significance check specifically (not recomputed here).
    below_shuffle_significance = (
        (~below_cc_floor) & (~below_cohen_d) & active_cells & pattern_reliable & (~combined_reliable)
    )

    df = pd.DataFrame({
        'cell_idx': np.arange(n_cells),
        'combined_reliable': combined_reliable,
        'avg_cc': avg_cc,
        'cohen_d': cohen_d,
        'active': active_cells,
        'pattern_reliable': pattern_reliable,
        'odd_even_corr': odd_even_corr,
        'peak_distance': peak_distances,
        'below_cc_floor': below_cc_floor,
        'below_cohen_d': below_cohen_d,
        'inactive': inactive,
        'pattern_unstable': pattern_unstable,
        'below_shuffle_significance_inferred': below_shuffle_significance,
    })

    failed_df = df[~df['combined_reliable']]

    summary = {
        'n_failed': len(failed_df),
        'below_cc_floor': int(failed_df['below_cc_floor'].sum()),
        'below_cohen_d': int(failed_df['below_cohen_d'].sum()),
        'inactive': int(failed_df['inactive'].sum()),
        'pattern_unstable': int(failed_df['pattern_unstable'].sum()),
        'below_shuffle_significance_inferred': int(failed_df['below_shuffle_significance_inferred'].sum()),
    }

    print(f"Of {summary['n_failed']} cells failing combined_reliable "
          f"(a cell can appear in multiple categories):")
    print(f"  below_cc_floor (avg_cc < {min_cc_threshold}): {summary['below_cc_floor']}")
    print(f"  below_cohen_d (cohen_d < {cohen_threshold}): {summary['below_cohen_d']}")
    print(f"  inactive (fails activity threshold): {summary['inactive']}")
    print(f"  pattern_unstable (odd/even peak/corr, 'fixed' test): {summary['pattern_unstable']}")
    print(f"  below_shuffle_significance (inferred by elimination): "
          f"{summary['below_shuffle_significance_inferred']}")

    return df, summary


In [ ]:
# --- Try it on the real Day1 session ---
failure_df, failure_summary = diagnose_reliability_failure_reasons(TEST_SESSION_DIR, smi_result_test)

# Specifically: of the 208 "high_smi_but_unreliable" cells from the diagnostic
# above, what did they fail on?
high_smi_unreliable_idx = diagnostic_df.loc[
    diagnostic_df['discrepancy'] == 'high_smi_but_unreliable', 'cell_idx'
]
print("\nFailure breakdown for just the 'high_smi_but_unreliable' cells:")
print(failure_df.set_index('cell_idx').loc[high_smi_unreliable_idx,
      ['below_cc_floor', 'below_cohen_d', 'inactive', 'pattern_unstable',
       'below_shuffle_significance_inferred']].sum())


## Phase 0 sensitivity check (not part of the Phase 3 outline) — `pattern_test='shuffled'`

Bounded, reversible test of whether the flat `peak_distance_threshold=5`
bin floor is miscalibrated for this V1-prism DREADD data, per the discussion
above. Reruns *only* the reliability sub-step (`combined_reliability_test_improved`,
imported, unmodified) on the already-saved `spatial_activity` from
`preproc.h5` with `pattern_test='shuffled'` instead of the current
`'fixed'` default — does **not** touch `Preprocess.py` or redo dF/F/
alignment/spatial discretization. Every other parameter is kept identical
to what `Preprocess.py` already used for `area='V1'`, so only the pattern
test itself differs.

Takes a bit longer than a normal reliability check since the shuffled
pattern test runs its own `n_shuffles` circular-shift null per cell, on
top of the correlation test's own shuffles.

- **Input:** `session_dir`, `pattern_test='shuffled'`, plus the same
  parameters `Preprocess.py` used for `area='V1'` (kept as defaults here).
- **Output:** dict with `combined_reliable`, `reliable_cells`,
  `pattern_reliable`, `avg_cc`, `cohen_d`, `odd_even_corr`,
  `peak_distances`, `active_cells`.

In [ ]:
from helper.ReliabilityTesting import combined_reliability_test_improved


def run_reliability_sensitivity_check(session_dir, pattern_test='shuffled',
                                       n_shuffles=300, cc_percentile=90,
                                       cohen_threshold=0.8, min_cc_threshold=0.1,
                                       min_pattern_corr=0.3, peak_distance_threshold=5,
                                       pattern_percentile=95, peak_distance_percentile=95,
                                       use_activity_threshold=True,
                                       activity_method='absolute_percentile'):
    """
    Rerun just the reliability test (not the whole Preprocess.py pipeline)
    on this session's already-saved spatial_activity, with a chosen
    pattern_test setting. See markdown above.

    Parameters
    ----------
    session_dir : str
    pattern_test : str
        'shuffled' (the alternative being tested) or 'fixed' (rerun the
        current default for a same-call comparison baseline).
    Remaining parameters : same values Preprocess.py used for area='V1',
        so only pattern_test differs from what's already in preproc.h5.

    Returns
    -------
    result : dict -- see markdown above for keys.
    """
    preproc_files = glob.glob(os.path.join(session_dir, "*preproc*.h5"))
    if not preproc_files:
        raise FileNotFoundError(f"No *preproc*.h5 found in {session_dir}")
    preproc_data = files.read_h5(preproc_files[0])
    spatial_activity = preproc_data['spatial_activity']

    (combined_reliable, reliable_cells, pattern_reliable, avg_cc, cohen_d,
     odd_even_corr, peak_distances, active_cells) = combined_reliability_test_improved(
        spatial_activity,
        n_shuffles=n_shuffles,
        cc_percentile=cc_percentile,
        cohen_threshold=cohen_threshold,
        min_cc_threshold=min_cc_threshold,
        min_pattern_corr=min_pattern_corr,
        peak_distance_threshold=peak_distance_threshold,
        use_activity_threshold=use_activity_threshold,
        activity_method=activity_method,
        pattern_test=pattern_test,
        pattern_percentile=pattern_percentile,
        peak_distance_percentile=peak_distance_percentile,
    )

    return {
        'combined_reliable': combined_reliable,
        'reliable_cells': reliable_cells,
        'pattern_reliable': pattern_reliable,
        'avg_cc': avg_cc,
        'cohen_d': cohen_d,
        'odd_even_corr': odd_even_corr,
        'peak_distances': peak_distances,
        'active_cells': active_cells,
    }


In [ ]:
# --- Run the sensitivity check on the real Day1 session ---
# This will take a bit longer than the original reliability test (extra
# per-cell shuffled null for the pattern test on top of the correlation test).
sensitivity_shuffled = run_reliability_sensitivity_check(TEST_SESSION_DIR, pattern_test='shuffled')

original_reliable = smi_result_test['combined_reliable']
new_reliable = sensitivity_shuffled['combined_reliable']
newly_recovered = new_reliable & (~original_reliable)
lost = original_reliable & (~new_reliable)

print("\n=== Comparison: 'fixed' (original, in preproc.h5) vs 'shuffled' ===")
print(f"Original combined_reliable:  {np.sum(original_reliable)} / {len(original_reliable)}")
print(f"Shuffled combined_reliable:  {np.sum(new_reliable)} / {len(new_reliable)}")
print(f"Newly recovered:             {np.sum(newly_recovered)}")
print(f"Lost (was reliable, now not): {np.sum(lost)}")


In [ ]:
def plot_newly_recovered_cells(session_dir, newly_recovered_mask, n_examples=8, seed=0):
    """
    Plot odd-vs-even mean tuning curves for a random sample of newly-
    recovered cells, so you can eyeball whether they look like genuine
    place fields or noise before trusting the sensitivity check's counts
    alone.

    Parameters
    ----------
    session_dir : str
    newly_recovered_mask : numpy.ndarray of bool
    n_examples : int
    seed : int

    Returns
    -------
    fig : matplotlib.figure.Figure or None (if nothing to plot)
    """
    preproc_files = glob.glob(os.path.join(session_dir, "*preproc*.h5"))
    preproc_data = files.read_h5(preproc_files[0])
    spatial_activity = preproc_data['spatial_activity']
    bin_centers = preproc_data['bin_centers']

    idx = np.where(newly_recovered_mask)[0]
    if len(idx) == 0:
        print("No newly recovered cells to plot.")
        return None

    rng = np.random.default_rng(seed)
    sample = rng.choice(idx, size=min(n_examples, len(idx)), replace=False)

    n_trials = spatial_activity.shape[1]
    odd_trials = np.arange(0, n_trials, 2)
    even_trials = np.arange(1, n_trials, 2)

    n_cols = 4
    n_rows = int(np.ceil(len(sample) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for ax, cell in zip(axes, sample):
        odd_mean = np.mean(spatial_activity[cell, odd_trials], axis=0)
        even_mean = np.mean(spatial_activity[cell, even_trials], axis=0)
        ax.plot(bin_centers, odd_mean, label='odd trials', color='steelblue')
        ax.plot(bin_centers, even_mean, label='even trials', color='coral')
        ax.set_title(f"Cell {cell}")
        ax.set_xlabel('Position (cm)')
        ax.set_ylabel('Activity')
        ax.legend(fontsize=10)

    for ax in axes[len(sample):]:
        ax.axis('off')

    fig.suptitle(f"{len(sample)} randomly-sampled newly-recovered cells (odd vs even trials)",
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    return fig


# --- Eyeball a sample of the newly-recovered cells ---
fig = plot_newly_recovered_cells(TEST_SESSION_DIR, newly_recovered, n_examples=8)
plt.show()


## Function 3.4 — `join_smi_to_master_table`

Track A: joins a Phase 2 tracking group's `master_df` to each session's SMI
output, using the `roi_idx_<label>` columns `master_df` already has (Phase
2's `build_master_cell_table`) as the join key — same iscell-filtered-
position convention used throughout this whole pipeline, no translation
needed.

Takes `master_df` as a plain in-memory DataFrame rather than reloading
anything from Phase 2's saved `.h5` itself, keeping this function fully
decoupled from Phase 2's internals -- you get `master_df` from
`2.CellTracking.py`'s `track_and_build_table`/`build_master_cell_table`
(same kernel session, or however you carry it over) and hand it in as-is.

- **Input:** `master_df` (from Phase 2, needs `roi_idx_<label>` columns),
  `smi_results_by_label` (`{label: run_smi_analysis_session(...) result}`,
  one entry per session in the group).
- **Output:** `df` — `master_df` plus new `SMI_<label>`,
  `combined_reliable_<label>`, `analysis_reliable_<label>`, `avg_cc_<label>`
  columns per session.

**Testing this for real** needs SMI computed for all sessions in a group,
not just Day1 -- I'll also run it for the SALINE/DCZ sessions from the same
`Day1_vs_SalineDCZ1` group so this has a genuine 3-session join to validate
against, using a lightweight direct reload of Phase 2's saved tracking
group `.h5` (not a new permanent function -- just enough to test).

### Postscript, added while building Phase 4 — read before resuming Track A

Track B has no cross-session cell identity, so a Track B comparison can
never restrict to "cells reliable in both sessions" using each session's
own full data — every Track B comparison re-establishes "reliable"
independently per session, at that session's own trial count. Saline
recordings are shorter than DCZ recordings within each pair, so this can
bias BOTH the reliable-cell fraction and the SMI magnitude among survivors
(shorter/noisier sessions show survivorship bias toward only the most
robustly-tuned cells) — not just a power/N issue, a real confound.

Track A's paired within-cell comparison (`join_smi_to_master_table` above)
sidesteps this entirely, since the same tracked cell's SMI is compared
across sessions regardless of either session's trial count — it does NOT
need trial-matching to be valid, unlike Track B. This is a real reason to
treat Track A as the more decisive test of the saline-vs-DCZ question, not
just a "more sensitive" alternative to Track B's population-level
comparison. See `4.SessionComparison.ipynb`'s Function 4.5 (reliability-
fraction vs. trial-count calibration check) for the diagnostic that
surfaced this. This same note is duplicated in `3.SMICalculation.py`'s
module docstring.

In [ ]:
def join_smi_to_master_table(master_df, smi_results_by_label):
    """
    Join a Phase 2 tracking group's master_df to each session's SMI output.
    See markdown above.

    Parameters
    ----------
    master_df : pandas.DataFrame
        From Phase 2's build_master_cell_table (optionally already passed
        through flag_layer_mismatches).
    smi_results_by_label : dict
        {label: run_smi_analysis_session(...) result}.

    Returns
    -------
    df : pandas.DataFrame
    """
    df = master_df.copy()

    roi_idx_cols = [c for c in df.columns if c.startswith('roi_idx_')]
    labels = [c[len('roi_idx_'):] for c in roi_idx_cols]

    for label in labels:
        if label not in smi_results_by_label:
            print(f"WARNING: no SMI result provided for '{label}' -- skipping.")
            continue

        smi_result = smi_results_by_label[label]
        SMI_values = smi_result['results_full']['smi_results']['SMI']
        combined_reliable = smi_result['combined_reliable']
        avg_cc = smi_result['avg_cc']
        analysis_reliable_cells = smi_result['analysis_reliable_cells']

        roi_idx = df[f'roi_idx_{label}'].to_numpy()

        df[f'SMI_{label}'] = SMI_values[roi_idx]
        df[f'combined_reliable_{label}'] = combined_reliable[roi_idx]
        df[f'analysis_reliable_{label}'] = analysis_reliable_cells[roi_idx]
        df[f'avg_cc_{label}'] = avg_cc[roi_idx]

    n_reliable_all = np.ones(len(df), dtype=bool)
    for label in labels:
        if f'analysis_reliable_{label}' in df.columns:
            n_reliable_all &= df[f'analysis_reliable_{label}'].to_numpy()

    print(f"Joined SMI for {len(labels)} sessions onto {len(df)} tracked cells.")
    for label in labels:
        if f'analysis_reliable_{label}' in df.columns:
            print(f"  analysis_reliable in {label}: {df[f'analysis_reliable_{label}'].sum()}/{len(df)}")
    print(f"  analysis_reliable in ALL {len(labels)} sessions "
          f"(usable for a paired within-cell comparison): {n_reliable_all.sum()}/{len(df)}")

    return df


In [ ]:
# --- Test setup: reload Phase 2's saved Day1_vs_SalineDCZ1 group (lightweight, test-only) ---
group_save_path = os.path.join(TEST_SESSION_DIR, 'TrackedGroups', 'Day1_vs_SalineDCZ1_tracking_results.h5')

def _decode(x):
    return x.decode() if isinstance(x, bytes) else x

with h5py.File(group_save_path, 'r') as f:
    verified_matrix = f['verified_matrix'][:]
    labels_order = [_decode(l) for l in f.attrs['labels_order']]
    plane0_paths = {label: _decode(f['plane0_paths'].attrs[label]) for label in labels_order}

print("labels_order:", labels_order)

mask = np.all(verified_matrix >= 0, axis=1)
tracked_rows = np.where(mask)[0]

test_master_df = pd.DataFrame({'global_cell_id': tracked_rows})
for col, label in enumerate(labels_order):
    test_master_df[f'roi_idx_{label}'] = verified_matrix[tracked_rows, col]

print(f"\nMinimal test master_df: {len(test_master_df)} tracked cells")
test_master_df.head()


In [ ]:
# --- Run SMI for the other two sessions in the group (Day1 already done above) ---
# This runs the same full SMI pipeline (with all its figure-saving) two more
# times -- expect this to take a few minutes total, same as the Day1 run did.
smi_results_by_label = {'Day1': smi_result_test}

for label in labels_order:
    if label == 'Day1':
        continue
    session_dir = os.path.dirname(os.path.dirname(plane0_paths[label]))
    print(f"\n{'='*90}\nRunning SMI for: {label}\n{'='*90}")
    smi_results_by_label[label] = run_smi_analysis_session(session_dir, session_label=label)


In [ ]:
# --- The actual Function 3.4 test: join SMI onto the tracked-cell table ---
joined_df = join_smi_to_master_table(test_master_df, smi_results_by_label)
joined_df.head(10)


## Function 3.5 — Track B session picker + batch runner

From here on, Track A (tracking-based) work pauses — Function 3.4 stays as
built but unused for now. Focus shifts to finishing Track B across the
remaining phases first; Track A gets revisited across all phases after
Phase 7. Every Track-B function is still built so a later Track A pass can
reuse its per-session outputs untouched (already true of Function 3.2's
`*_smi_results_dreadd.h5`).

Built as three pieces:
1. **`discover_animal_sessions`** — reimplemented from Phase 2 (identical
   logic; digit-prefixed module filenames can't be imported).
2. **`select_sessions_popup`** — same checkbox-popup UI as Phase 2's, but
   Track B needs no reference/grouping choice at all — just pick however
   many sessions you want SMI run on, each fully independently.
3. **`run_smi_batch_track_b`** — loops Function 3.2 over the checked
   sessions, collecting one result per session.

- **Input:** `animal_dir` (for discovery); then `session_catalog` +
  `selected_labels` (from the picker) for the batch run, plus
  `run_smi_analysis_session`'s usual parameters.
- **Output:** `{label: run_smi_analysis_session(...) result}`.

In [ ]:
def discover_animal_sessions(animal_dir):
    """
    Scan animal_dir for every TSeries-*/suite2p/plane0 folder anywhere in
    its tree, labeling each by whichever known naming pattern it matches.

    Unlike an earlier version of this function, nothing is silently
    dropped on a label collision (e.g. a day with both a NO_REWARD probe
    and a normal recording, or a session split into sal-001/sal-002
    parts) -- every colliding entry is disambiguated by appending its raw
    TSeries folder name, so all of them show up in the picker and you
    choose which one(s) belong.

    Parameters
    ----------
    animal_dir : str

    Returns
    -------
    catalog : dict
        {label: {'plane0_path': str, 'session_type': str, 'tseries_dir': str}}
    """
    plane0_paths = sorted(glob.glob(os.path.join(animal_dir, '**', 'suite2p', 'plane0'),
                                     recursive=True))

    entries = []  # (base_label, session_type, plane0_path, tseries_dir, tseries_name)
    skipped = []
    unmatched = []

    for plane0_path in plane0_paths:
        tseries_dir = os.path.dirname(os.path.dirname(plane0_path))
        tseries_name = os.path.basename(tseries_dir)
        parent_dir = os.path.dirname(tseries_dir)
        parent_name = os.path.basename(parent_dir)

        if 'skip' in tseries_name.lower() or 'skip' in parent_name.lower():
            skipped.append(tseries_dir)
            continue

        upper_tseries = tseries_name.upper()

        if 'SAL' in upper_tseries:
            session_type = 'saline'
            base_label = f'{parent_name}_SALINE'
        elif 'DCZ' in upper_tseries:
            session_type = 'dcz'
            base_label = f'{parent_name}_DCZ'
        else:
            day_match = re.search(r'Day(\d+)', parent_name, re.IGNORECASE)
            if day_match:
                session_type = 'baseline'
                base_label = f'Day{day_match.group(1)}'
            else:
                session_type = 'unknown'
                base_label = tseries_name
                unmatched.append(base_label)

        entries.append((base_label, session_type, plane0_path, tseries_dir, tseries_name))

    # Disambiguate every base_label that collides -- suffix ALL of its
    # entries with the raw TSeries folder name (not just the extras), so
    # the labeling is consistent whether or not a collision happened.
    from collections import Counter
    label_counts = Counter(e[0] for e in entries)

    catalog = {}
    for base_label, session_type, plane0_path, tseries_dir, tseries_name in entries:
        if label_counts[base_label] > 1:
            label = f'{base_label}__{tseries_name}'
        else:
            label = base_label

        if label in catalog:
            print(f"WARNING: label '{label}' still collides after disambiguation -- "
                  f"keeping {catalog[label]['tseries_dir']}, skipping {tseries_dir}")
            continue

        catalog[label] = {
            'plane0_path': plane0_path,
            'session_type': session_type,
            'tseries_dir': tseries_dir,
        }

    print(f"Discovered {len(catalog)} sessions under {animal_dir}:")
    for label, info in catalog.items():
        print(f"  [{info['session_type']:>8}] {label}  <-  {info['tseries_dir']}")

    collided_labels = [l for l, c in label_counts.items() if c > 1]
    if collided_labels:
        print(f"\n{len(collided_labels)} label(s) had multiple TSeries and were disambiguated "
              f"-- review these and pick the right one(s) in the popup: {collided_labels}")

    if skipped:
        print(f"\n{len(skipped)} session(s) excluded via 'skip' in their folder name:")
        for s in skipped:
            print(f"  {s}")

    if unmatched:
        print(f"\n{len(unmatched)} session(s) didn't match a known naming pattern "
              f"(labeled 'unknown', tagged by raw TSeries folder name): {unmatched}")

    return catalog


def select_sessions_popup(session_catalog,
                           title='Select sessions to run SMI on (Track B -- independent per session)'):
    """
    Checkbox popup listing every session in session_catalog. Same UI as
    Phase 2's function of the same name. Track B needs no reference/
    grouping choice -- just pick however many sessions you want SMI run on,
    each fully independently.

    Parameters
    ----------
    session_catalog : dict
        From discover_animal_sessions.
    title : str

    Returns
    -------
    selected_labels : list of str
        Labels you checked, in session_catalog's original order.
    """
    labels = list(session_catalog.keys())
    n = len(labels)
    display_labels = [f"[{session_catalog[l]['session_type']:>8}] {l}" for l in labels]

    fig_height = max(4, 0.35 * n + 1.5)
    fig = plt.figure(figsize=(9, fig_height))
    try:
        fig.canvas.manager.set_window_title('SELECT SESSIONS (Track B) -- check boxes, then Confirm (or Enter)')
    except Exception:
        pass

    fig.suptitle(title, fontsize=12, fontweight='bold')

    check_ax = fig.add_axes([0.05, 0.12, 0.9, 0.80])
    check = CheckButtons(check_ax, display_labels, [False] * n)

    confirm_ax = fig.add_axes([0.35, 0.02, 0.3, 0.06])
    confirm_button = Button(confirm_ax, 'Confirm selection')

    state = {'done': False, 'status': [False] * n}

    def on_confirm(event=None):
        state['status'] = list(check.get_status())
        state['done'] = True
        fig.canvas.stop_event_loop()

    confirm_button.on_clicked(on_confirm)

    def on_key(event):
        if event.key == 'enter':
            on_confirm()

    fig.canvas.mpl_connect('key_press_event', on_key)

    plt.show(block=False)
    while not state['done'] and plt.fignum_exists(fig.number):
        fig.canvas.start_event_loop(0.1)

    if not state['done']:
        state['status'] = list(check.get_status())
        print("Window closed without pressing Confirm -- using current checkbox state anyway.")

    if plt.fignum_exists(fig.number):
        plt.close(fig)
        plt.pause(0.01)  # let Qt actually process the close/hide before we return

    selected_labels = [label for label, checked in zip(labels, state['status']) if checked]
    print(f"Selected {len(selected_labels)} sessions: {selected_labels}")

    return selected_labels


def run_smi_batch_track_b(session_catalog, selected_labels, **smi_kwargs):
    """
    Loop run_smi_analysis_session over the checked sessions, each fully
    independently (Track B).

    Parameters
    ----------
    session_catalog : dict
        From discover_animal_sessions.
    selected_labels : list of str
        From select_sessions_popup (or any list of labels you already have).
    **smi_kwargs
        Passed through to run_smi_analysis_session (e.g. segment_distance,
        exclude_start_cm, ...).

    Returns
    -------
    results_by_label : dict
        {label: run_smi_analysis_session(...) result}.
    """
    results_by_label = {}

    for label in selected_labels:
        session_dir = os.path.dirname(os.path.dirname(session_catalog[label]['plane0_path']))
        print(f"\n{'='*90}\nTRACK B -- {label}\n{'='*90}")
        results_by_label[label] = run_smi_analysis_session(
            session_dir, session_label=label, **smi_kwargs
        )

    print(f"\n{'='*90}\nBatch complete: {len(results_by_label)}/{len(selected_labels)} sessions")
    for label, result in results_by_label.items():
        n_reliable = np.sum(result['analysis_reliable_cells'])
        n_total = len(result['analysis_reliable_cells'])
        print(f"  {label}: {n_reliable}/{n_total} analysis-reliable cells")

    return results_by_label


In [ ]:
# --- Test Function 3.5: discover -> pick -> batch run (Track B) ---
session_catalog = discover_animal_sessions(TEST_ANIMAL_DIR)

selected_labels = select_sessions_popup(session_catalog)

# track_b_results = run_smi_batch_track_b(session_catalog, selected_labels)
